In [1]:
# [Cell 0] Install Dependencies
# [Cell 0] Install Dependencies
!pip install -q \
    "opentelemetry-api>=1.39.0,<=1.42.1" \
    "opentelemetry-sdk>=1.39.0,<=1.42.1" \
    chromadb \
    sentence-transformers \
    pypdf \
    python-docx

In [2]:
# [Cell 1] Vector Embedding Intuition
from sentence_transformers import SentenceTransformer
import numpy as np

# Load a lightweight, industry-standard sentence embedding model
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# Three sentences: two share meaning, one is unrelated
sentences = [
    "The physician examined the patient.",
    "A doctor checked the sick individual.",
    "The sports car drove down the highway."
]

# Generate dense vector embeddings (384 floating-point numbers each)
embeddings = embed_model.encode(sentences)

print(f"Embedding shape for each sentence: {embeddings[0].shape}")
print(f"First 5 dimensions of sentence 1:\n{embeddings[0][:5]}\n")

# Define Cosine Similarity calculation
def cosine_similarity(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

sim_1_2 = cosine_similarity(embeddings[0], embeddings[1])
sim_1_3 = cosine_similarity(embeddings[0], embeddings[2])

print(f"Similarity between 'physician' and 'doctor' sentences: {sim_1_2:.4f}")
print(f"Similarity between 'physician' and 'sports car' sentences: {sim_1_3:.4f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding shape for each sentence: (384,)
First 5 dimensions of sentence 1:
[ 0.02281482  0.11275595 -0.06518739 -0.00805107 -0.08706454]

Similarity between 'physician' and 'doctor' sentences: 0.5672
Similarity between 'physician' and 'sports car' sentences: -0.0080


In [3]:
# [Cell 2] Initialize ChromaDB
import chromadb
from chromadb.utils import embedding_functions

# 1. Instantiate an in-memory client
chroma_client = chromadb.Client()

# 2. Configure Chroma to automatically use all-MiniLM-L6-v2 for all additions and queries
st_embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# 3. Create a collection (equivalent to a table in SQL)
collection = chroma_client.get_or_create_collection(
    name="medicore_knowledge",
    embedding_function=st_embedding_function
)

print(f"Collection '{collection.name}' initialized successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Collection 'medicore_knowledge' initialized successfully.


In [4]:
# [Cell 3a] Download the dataset if not already present
!wget -nc -q https://raw.githubusercontent.com/AI-Learning-Repo/Data-Handling/refs/heads/week4/datasets/MediCore.json

In [5]:
# [Cell 3b] Ingest Data into ChromaDB
import json

# 1. Read the JSON file
with open("MediCore.json", "r", encoding="utf-8") as f:
    lines = f.readlines()

documents = []
metadatas = []
ids = []

for idx, line in enumerate(lines):
    item = json.loads(line.strip())

    # Store the factual completion as the searchable text
    documents.append(item["completion"])

    # Extract simple metadata: associate the question prompt as context
    metadatas.append({"source_prompt": item["prompt"], "doc_index": idx})
    ids.append(f"medicore_fact_{idx}")

# 2. Add records to the vector collection in a single batch
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print(f"Successfully indexed {collection.count()} document chunks into ChromaDB.")

Successfully indexed 486 document chunks into ChromaDB.


In [6]:
# [Cell 4] Query ChromaDB
query_text = "Who is in charge of brain and nervous system conditions?"

# Retrieve the top 2 closest semantic matches
results = collection.query(
    query_texts=[query_text],
    n_results=2
)

# Inspect the returned results
print(f"QUERY: {query_text}\n")
for i in range(len(results["documents"][0])):
    doc = results["documents"][0][i]
    distance = results["distances"][0][i]
    doc_id = results["ids"][0][i]
    print(f"Rank {i+1} [Distance: {distance:.4f}] [ID: {doc_id}]:")
    print(f"Content: {doc}\n")

QUERY: Who is in charge of brain and nervous system conditions?

Rank 1 [Distance: 0.4165] [ID: medicore_fact_30]:
Content: The neurology department handles brain and nervous system diseases at MediCore Hospital.

Rank 2 [Distance: 0.5325] [ID: medicore_fact_56]:
Content: Dr. Elena Varga leads the neurology department at MediCore Hospital.



In [13]:
# [Cell 5] Instant Knowledge Update (CEO Leadership Transition)

target_id = "medicore_fact_71"
query = "Who is the CEO of MediCore Hospital?"

# Step 1: Ensure the record is set to the baseline value (allows clean re-runs)
collection.update(
    ids=[target_id],
    documents=["The CEO of MediCore Hospital is Juhani Aho."]
)

# Step 2: Query the database before modification
print("--- BEFORE UPDATE ---")
search_before = collection.query(query_texts=[query], n_results=1)
doc_before = search_before["documents"][0][0]
dist_before = search_before["distances"][0][0]
print(f"Rank 1 [Distance: {dist_before:.4f}]: {doc_before}")

# Step 3: Mutate the record in the vector database
# Scenario: Juhani Aho retires; Milla Kallio is appointed as the new CEO
collection.update(
    ids=[target_id],
    documents=["The CEO of MediCore Hospital is Milla Kallio."]
)

# Step 4: Query the database after modification
print("\n--- AFTER UPDATE ---")
search_after = collection.query(query_texts=[query], n_results=1)
doc_after = search_after["documents"][0][0]
dist_after = search_after["distances"][0][0]
print(f"Rank 1 [Distance: {dist_after:.4f}]: {doc_after}")

--- BEFORE UPDATE ---
Rank 1 [Distance: 0.1759]: The CEO of MediCore Hospital is Juhani Aho.

--- AFTER UPDATE ---
Rank 1 [Distance: 0.1392]: The CEO of MediCore Hospital is Milla Kallio.


In [8]:
# [Appendix Cell A] PDF Extraction Proof-of-Concept
from pypdf import PdfReader
import io

# 1. Create a minimal in-memory PDF for demonstration purposes
# (In practice, you would pass a path like: reader = PdfReader("hospital_policy.pdf"))
from pypdf import PdfWriter
writer = PdfWriter()
writer.add_blank_page(width=200, height=200)
pdf_stream = io.BytesIO()
writer.write(pdf_stream)
pdf_stream.seek(0)

# 2. Extract text page-by-page
def extract_text_from_pdf(file_source):
    reader = PdfReader(file_source)
    extracted_text = []

    for page_num, page in enumerate(reader.pages):
        text = page.extract_text()
        if text:
            extracted_text.append(text)

    return "\n".join(extracted_text)

# Example usage:
# full_text = extract_text_from_pdf("my_policy.pdf")
print("PDF extraction function defined successfully.")

PDF extraction function defined successfully.


In [9]:
# [Appendix Cell B] DOCX Extraction Proof-of-Concept
import docx

def extract_text_from_docx(file_path):
    doc = docx.Document(file_path)
    full_text = []

    # Extract text from every paragraph
    for para in doc.paragraphs:
        if para.text.strip():  # Skip empty lines
            full_text.append(para.text.strip())

    return "\n".join(full_text)

print("DOCX extraction function defined successfully.")

DOCX extraction function defined successfully.


In [10]:
# [Appendix Cell C] Simple Text Chunking Strategy
sample_long_document = """
MediCore Hospital Emergency Protocol:
All patients arriving with acute chest pain must undergo an immediate ECG within 10 minutes of arrival.
The triage nurse must assign an emergency severity index (ESI) of level 2 or higher.
The attending cardiologist on duty must be notified immediately via the direct emergency line.
Blood samples for cardiac troponin testing must be drawn at bedside upon triage completion.
"""

def chunk_text(text, chunk_size=150, overlap=30):
    """Splits text into chunks of roughly chunk_size characters with overlap."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += (chunk_size - overlap)
    return chunks

chunks = chunk_text(sample_long_document, chunk_size=120, overlap=20)

print(f"Divided long text into {len(chunks)} chunks:")
for i, c in enumerate(chunks):
    print(f"Chunk {i+1}: {c}")

Divided long text into 5 chunks:
Chunk 1: MediCore Hospital Emergency Protocol:
All patients arriving with acute chest pain must undergo an immediate ECG within
Chunk 2: mmediate ECG within 10 minutes of arrival.
The triage nurse must assign an emergency severity index (ESI) of level 2 or
Chunk 3: (ESI) of level 2 or higher.
The attending cardiologist on duty must be notified immediately via the direct emergency lin
Chunk 4: direct emergency line.
Blood samples for cardiac troponin testing must be drawn at bedside upon triage completion.
Chunk 5: ge completion.


In [12]:
# [Appendix Cell] End-to-End Proof of Concept: Unstructured Files (DOCX & PDF) to ChromaDB
!pip install -q reportlab

import os
import docx
from pypdf import PdfReader
from reportlab.pdfgen import canvas
import chromadb
from chromadb.utils import embedding_functions

# =====================================================================
# Step 1: Create sample DOCX and PDF files directly in Colab
# =====================================================================

# 1a. Create sample Word Document
doc_path = "hospital_policy.docx"
doc = docx.Document()
doc.add_heading("MediCore Hospital Acute Care Protocols", level=1)
doc.add_paragraph(
    "All patients arriving with acute chest pain must receive an immediate 12-lead ECG "
    "within 10 minutes of registration. The attending cardiologist must be paged immediately."
)
doc.add_paragraph(
    "Emergency stroke patients require an immediate non-contrast head CT scan. "
    "Thrombolytic therapy must be evaluated within 45 minutes of door arrival."
)
doc.save(doc_path)

# 1b. Create sample PDF Document
pdf_path = "surgical_protocols.pdf"
c = canvas.Canvas(pdf_path)
c.drawString(72, 750, "MediCore Hospital Surgical Division Policy:")
c.drawString(72, 730, "Robotic-assisted surgery suites require full UV-C terminal sterilization.")
c.drawString(72, 710, "Surgeons must complete 3D virtual simulation before operating with the MediBot.")
c.save()

print("Generated sample files: hospital_policy.docx, surgical_protocols.pdf")

# =====================================================================
# Step 2: Extraction Functions
# =====================================================================

def extract_from_docx(file_path):
    d = docx.Document(file_path)
    paragraphs = [p.text.strip() for p in d.paragraphs if p.text.strip()]
    return "\n".join(paragraphs)

def extract_from_pdf(file_path):
    reader = PdfReader(file_path)
    pages_text = []
    for page in reader.pages:
        text = page.extract_text()
        if text:
            pages_text.append(text.strip())
    return "\n".join(pages_text)

docx_text = extract_from_docx(doc_path)
pdf_text = extract_from_pdf(pdf_path)

# =====================================================================
# Step 3: Fixed-Size Text Chunking with Overlap
# =====================================================================

def chunk_text(text, source_name, chunk_size=150, overlap=30):
    chunks = []
    metadatas = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
            metadatas.append({"source": source_name, "char_start": start})
        start += (chunk_size - overlap)
    return chunks, metadatas

docx_chunks, docx_meta = chunk_text(docx_text, source_name="hospital_policy.docx")
pdf_chunks, pdf_meta = chunk_text(pdf_text, source_name="surgical_protocols.pdf")

all_chunks = docx_chunks + pdf_chunks
all_metadatas = docx_meta + pdf_meta
all_ids = [f"unstructured_chunk_{i}" for i in range(len(all_chunks))]

# =====================================================================
# Step 4: Ingest into a Dedicated ChromaDB Collection
# =====================================================================

chroma_client = chromadb.Client()
st_embed = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

unstructured_collection = chroma_client.create_collection(
    name="unstructured_demo",
    embedding_function=st_embed
)

unstructured_collection.add(
    documents=all_chunks,
    metadatas=all_metadatas,
    ids=all_ids
)

print(f"Indexed {unstructured_collection.count()} chunks from DOCX and PDF into ChromaDB.\n")

# =====================================================================
# Step 5: Query Across Document Types
# =====================================================================

query = "What is the procedure for emergency chest pain?"

results = unstructured_collection.query(
    query_texts=[query],
    n_results=1
)

retrieved_doc = results["documents"][0][0]
source_file = results["metadatas"][0][0]["source"]
distance = results["distances"][0][0]

print(f"QUERY: {query}")
print(f"MATCH FROM SOURCE: [{source_file}] (Distance: {distance:.4f})")
print(f"CONTENT: {retrieved_doc}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 67.0 MB/s eta 0:00:00
Generated sample files: hospital_policy.docx, surgical_protocols.pdf
Indexed 5 chunks from DOCX and PDF into ChromaDB.

QUERY: What is the procedure for emergency chest pain?
MATCH FROM SOURCE: [hospital_policy.docx] (Distance: 0.4306)
CONTENT: MediCore Hospital Acute Care Protocols
All patients arriving with acute chest pain must receive an immediate 12-lead ECG within 10 minutes of registra
